In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('/Users/phoom/Desktop/kaggle/Churn Classification/ChurnModelling.csv')

In [7]:
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [11]:
# เช็คข้อมูลสูญหาย
df.isna().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

# Step 2: Data Cleaning & Feature Selection

คอลัมน์ `RowNumber`, `CustomerId`, และ `Surname` เป็นเพียงข้อมูลระบุตัวตน ไม่ได้มีผลต่อการตัดสินใจเลิกใช้บริการของลูกค้า เราจึงทำการลบออก

In [ ]:
# ลบคอลัมน์ที่ไม่จำเป็นออก
df_cleaned = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
print("คอลัมน์ที่เหลืออยู่:")
print(df_cleaned.columns.tolist())

# Step 3: Encoding Categorical Features

แปลงข้อมูลตัวอักษร (`Geography` และ `Gender`) ให้เป็นตัวเลขด้วย One-Hot Encoding (`pd.get_dummies`)

In [ ]:
# แปลงข้อมูลหมวดหมู่ด้วย One-Hot Encoding
df_encoded = pd.get_dummies(df_cleaned, columns=['Geography', 'Gender'], drop_first=True)
df_encoded.head()

# Step 4: Separate Features (X) / Target (y) & Train-Test Split

แยกข้อมูลเป็นตัวแปรต้น (X) และตัวแปรตาม (y) จากนั้นแบ่งเป็น ชุดฝึกสอน (80%) และ ชุดทดสอบ (20%)

In [ ]:
from sklearn.model_selection import train_test_split

# แยก X (Features) และ y (Target)
X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

# แบ่ง Train (80%) และ Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"จำนวนข้อมูล X_train: {X_train.shape}")
print(f"จำนวนข้อมูล X_test:  {X_test.shape}")

# Step 5: Feature Scaling

ปรับสเกลของตัวเลขให้อยู่ในช่วงมาตรฐานเดียวกันโดยใช้ `StandardScaler`

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("ตัวอย่างข้อมูลที่ปรับสเกลแล้ว:")
print(X_train_scaled[:2])

# Step 6: Model Training & Evaluation

สร้างโมเดล Random Forest มาฝึกสอน และประเมินผลประสิทธิภาพการทำนายด้วย `classification_report` และ `confusion_matrix`

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. สร้างและฝึกสอนโมเดล Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 2. ทำนายผลกับชุดข้อมูลทดสอบ (X_test_scaled)
y_pred = model.predict(X_test_scaled)

# 3. ประเมินผลลัพธ์
print(f"--- ความแม่นยำ (Accuracy): {accuracy_score(y_test, y_pred):.4f} ---\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))